In [ ]:
# Use kernal base (Python 3.13.5)
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
import json
import os
import time
import pprint
import pandas as pd 
from datetime import datetime

print(f"Loading .env from: {find_dotenv()}")
load_dotenv()  # Load environment variables from .env file

Loading .env from: /Users/prernaroy/Library/CloudStorage/OneDrive-UNSW/Year 2/disso/dissertation/.env


True

In [23]:
# === Configuration ===
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Load the API key from the .env file
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running this notebook.")

client = OpenAI(api_key=OPENAI_API_KEY)

# Full Variable List Testing

Managed to get the output into JSON reliably for 4 variables. This will now be extended to more variables, including some text summary variables to see what the output is like. 

In [24]:
# Pydantic model for structured data extraction of Coronial findings and recommendations.
class VariablesToExtractExtended(BaseModel):
    file_number: str
    mental_disorder_flag: bool
    mental_disorder_labels: list[str]
    cognitive_disability_flag: bool
    cognitive_disability_labels: list[str]
    date_of_findings: str
    date_of_death: str
    age_at_death: int # Potential change to str for age estimates
    sex: str
    country_of_birth: str
    place_of_residence: str
    profession: str
    level_of_education: str
    comorbidities: list[str]
    medications: list[str]
    known_health_issues: list[str]
    place_of_death: str
    definitive_cause_of_death: str
    manner_of_death: str
    site_of_incident: str
    potential_causes_of_death: list[str]
    coroners_recommendations_summary: str
    custody_served: bool
    use_of_interpreters: bool
    preventable_death_flag: bool
    non_english_speaking_background_flag: bool
    circumstances_of_death_summary: str
    toxicology_results_summary: str
    domestic_violence_flag: bool

## Prompt Cells

In [25]:
instruction = ""
# Compare to just asking straightforwardly without defining tone/goal 

In [26]:
system_content = "You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure."
# Move this into instructions? 
# Include a sentence here to specify what is a MHC and DD

In [27]:
user_text = """Extract the following medical and demographic information from the given coronial findings and recommmendations report. Only extract information present in the report. If a field is not present or unknown, return an empty string, an empty list for list fields, and False for boolean flags. Flags must be True if verifiably present in the report with supporting evidence such as doctor diagnoses, or stated clearly in the inquest report from a verifiable source. Outputs must be accurate and concise; do not provide additional commentary or explanation. Labels should be short, specific, and present in the report; do not paraphrase directly from the report. For summary variables, provide a summary of the key points if present, otherwise return an empty string. Do not provide potential speculative information when definitive information is in the report. Do not provide additional information beyond what is requested. Do not repeat information that is already in another variable. Use ICD-10 to accurately identify mental disorders and cognitive disabilities.

Extract the following fields: file_number, mental_disorder_flag, mental_disorder_labels, cognitive_disability_flag, cognitive_disability_labels, date_of_findings, date_of_death, age_at_death, sex, country_of_birth, place_of_residence, profession, level_of_education, comorbidities, medications, known_health_issues, place_of_death, definitive_cause_of_death, manner_of_death, site_of_incident, potential_causes_of_death, coroners_recommendations_summary, custody_served, use_of_interpreters, preventable_death_flag, non_english_speaking_background_flag, circumstances_of_death_summary, toxicology_results_summary, and domestic_violence_flag."""

# Include definition of each field in model prompt and what is the requirement of this field. have a very clear list of what to extract 
# Compare current prompt with no definitions/requirements and zero-shot with explanations for each field as well to see change in performance 

# Research whether providing just sentence or the whole report (should be whole report) is sufficient for examples in few-shot 
# Few examples where I only give the sentence 
# Systematic approach on how to test this -- research 
# Reference Ida's dissertation - used whole discharge summary instead of just specific sections
# 

In [28]:
file_url = "https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf"
# file_url = "https://coroners.nsw.gov.au/documents/findings/2026/Inquest_into_the_death_of_BS.pdf"

Using this example, I see that there is an issue that needs to be addressed: the accurate definition of what is considered a mental disorder versus a developmental or cognitive disability. In different runs of this input, the output will sometimes only mark `cognitive_disability_flag` as `True` and `mental_disorder_flag` as `False`. However, in other runs, it considers ASD and ADHD to be mental disorders, while the only cognitive disability is "global developmental delay". 

This is something to think about, but for now I will focus on building the evaluation pipeline. For now I am creating my own evaluation pipeline, but it should be noted that there is a library called Promptfoo, which is now OpenAI's preferred way to conduct evaluations: https://developers.openai.com/cookbook/examples/evaluation/moving-from-openai-evals-to-promptfoo

In [29]:
# Consider including instructions in the response 
# response = ...(
#     instructions = "You are a helpful AI assistant"
# )

# instructions parameter gives high-level persistent behavioural guidelines such as role, tone, goals without overcrowding the main prompt

## Model Run

In [30]:
client = OpenAI()

# Define the parsing request to extract structured data from a coronial finding and recommendations report.
start_time = time.time()
response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    input=[
        {
            "role": "system",
            "content": system_content,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": user_text
                },
                {
                    "type": "input_file",
                    "file_url": file_url,
                }
            ]
        }

    ],
    text_format=VariablesToExtractExtended,
)
latency = time.time() - start_time

report_variables_json = response.output_parsed.model_dump_json(indent=2)
# print(report_variables_json)

# Few-shot examples to specify what is MHC and disability 
# Check out Aaron's dissertation - he specifies in the prompt; use as an example of "do this and not that"

In [79]:
# # Explore response object
# # print(response.model_dump_json(indent=2))
# resp_outputs = {
#     "response_id": response.id,
#     "created_at": response.created_at,
#     "completed_at": response.completed_at,
#     "instructions": response.instructions,
#     "metadata": response.metadata,
#     "model": response.model,
#     "temperature": response.temperature,
#     "top_p": response.top_p,
#     "max_output_tokens": response.max_output_tokens,
#     "usage_input_tokens": response.usage.input_tokens,
#     "usage_output_tokens": response.usage.output_tokens,
#     "usage_total_tokens": response.usage.total_tokens,
#     "output": response.output_text
# } 

# pprint.pprint(resp_outputs)


# # Keys to store from response object
# # prompt_response_log_objects = id, created_at, model, temperature, top_p, max_output_tokens, usage - input_tokens, usage - output_tokens, usage - total_tokens

In [74]:
# Pydantic Model to Store outputs and runtime information
class InfoLog(BaseModel):
    response_id: str
    created_at: str
    completed_at: str
    latency_seconds: float
    instructions: str | None
    file_url: str
    system_content: str
    user_text: str
    metadata: dict
    model: str | None
    temperature: float
    top_p: float 
    max_output_tokens: int | None
    usage_input_tokens: int
    usage_output_tokens: int
    usage_total_tokens: int
    output: dict
    

In [80]:
log_data = InfoLog(
    response_id = response.id,
    created_at = str(datetime.fromtimestamp(response.created_at)),
    completed_at = str(datetime.fromtimestamp(response.completed_at)),
    latency_seconds = round(latency, 4),
    instructions = response.instructions,
    file_url = file_url,
    system_content = system_content,
    user_text = user_text,
    metadata = response.metadata,
    model = response.model,
    temperature = response.temperature,
    top_p = response.top_p,
    max_output_tokens = response.max_output_tokens,
    usage_input_tokens = response.usage.input_tokens,
    usage_output_tokens = response.usage.output_tokens,
    usage_total_tokens = response.usage.total_tokens,
    output = json.loads(response.output_text)
)

In [81]:
log_data.model_dump_json

<bound method BaseModel.model_dump_json of InfoLog(response_id='resp_0cf5e5ac40193799006a7b3b27155c8198bd68ddd654475081', created_at='2026-08-12 01:09:29', completed_at='2026-08-12 01:09:34', latency_seconds=8.4779, instructions=None, file_url='https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf', system_content='You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure.', user_text='Extract the following medical and demographic information from the given coronial findings and recommmendations report. Only extract information present in the report. If a field is not present or unknown, return an empty string, an empty list for list fields, and False for boolean flags. Flags must be True if verifiably present in the report with supporting evidence such as doctor diagnoses, or stated clearly in the inquest re

In [83]:
with open("audit_log.jsonl", "a") as f:
    f.write(log_data.model_dump_json() + "\n")

In [60]:
# resp_outputs
# response.output_text

In [61]:
# # Create a JSONL file that stores the response_id, system_content, user_text, file_url called prompt_version_info
# prompt_info_version = {
#     "response_id":response.id,
#     "file_url":file_url,
#     "system_content":system_content,
#     "user_text":user_text
#     }

# prompt_info_version

In [ ]:
# # Append to a given jsonl file
# def append_to_jsonl(data_dict, filename="output.jsonl"):
#      with open(filename, "a", encoding="utf-8") as f:
#         f.write(json.dumps(data_dict) + "\n")

# # Need to add a numbering order index (i.e. 1, 2, 3)

In [ ]:
# append_to_jsonl(resp_outputs, filename="resp_outputs.jsonl")
# append_to_jsonl(prompt_info_version, filename="prompt_info_version.jsonl")

## Evaluation Pipeline Setup

The goal of the evaluation pipeline is to:
1. Version prompts and record the model output for each prompt used (store the prompt and the model output for future reference).
2. Determine the formatting accuracy of the model output (it should be JSON). 
3. Measure the accuracy and relevancy of the model output (the value produced for each variable extracted). 

### Binary Variable Evaluation
Entity-matching performance will be evaluated using standard classification metrics such as accuracy, precision, recall, negative predictive value (NPV) and F1-score.

In [34]:
type(json.loads(response.output_text))

dict

In [ ]:
output_dict = json.loads(response.output_text)
output_dict
# Manual evaluation process as the main evaluation technique -- more accurate 

{'file_number': '2017/00335331',
 'mental_disorder_flag': False,
 'mental_disorder_labels': [],
 'cognitive_disability_flag': True,
 'cognitive_disability_labels': ['global development delay'],
 'date_of_findings': '21 October 2022',
 'date_of_death': '5 November 2017',
 'age_at_death': 8,
 'sex': 'Male',
 'country_of_birth': '',
 'place_of_residence': 'Woy Woy, NSW',
 'profession': '',
 'level_of_education': '',
 'comorbidities': ['autism spectrum disorder',
  'attention deficit hyperactivity disorder'],
 'medications': ['amphetamine', 'risperidone', 'citalopram'],
 'known_health_issues': ['moderate to severe global development delay',
  'ASD-3',
  'limited speech',
  'predominantly non-verbal',
  'required close supervision',
  'behaviours of concern when travelling in vehicles'],
 'place_of_death': 'M1 Pacific Motorway near Cameron Park NSW',
 'definitive_cause_of_death': 'multiple injuries sustained when hit by a truck',
 'manner_of_death': 'Accidental',
 'site_of_incident': 'M1 Mo

In [47]:
test1_df = pd.read_csv("test1.csv")
test1_dict = test1_df.to_dict(orient="records")

In [48]:
test1_dict = test1_dict[0]

In [ ]:
test1_dict
# Try address evaluation using Bertscore -- show next week 

{'file_number': '2017/00335331',
 'mental_disorder_flag': False,
 'mental_disorder_labels': nan,
 'cognitive_disability_flag': True,
 'cognitive_disability_labels': 'Autism Spetrum Disorder, Attention-Deficit/Hyperactivity Disorder, Global Developmental Delay',
 'date_of_findings': '21/10/2022',
 'date_of_death': '5/11/2017',
 'age_at_death': 8,
 'sex': 'Male',
 'country_of_birth': nan,
 'place_of_residence': 'Woy Woy, NSW',
 'profession': nan,
 'level_of_education': nan,
 'comorbidities': nan,
 'medications': 'amphetamine, risperidone, citalopram',
 'known_health_issues': 'Mostly non-verbal with limited speech and communication skills',
 'place_of_death': 'M1 Pacific Motorway near Cameron Park\nNSW',
 'definitive_cause_of_death': 'Multiple injuries sustained when he was\nhit by a truck',
 'manner_of_death': "Riley died while escaping from the care of Rachel\nMartin, the carer employed by SNAP Programs to care\nfor him while in respite care. It is likely that Riley\nremoved a car seat 

In [53]:
output_dict.keys() == test1_dict.keys()

True

In [54]:
# List of exact-match/binary/boolean variables
exact_match_variables = [
    "file_number",
    "date_of_findings",
    "sex"
]

boolean_variables = [
    "mental_disorder_flag",
    "cognitive_disability_flag",
    "preventable_death_flag",
    "use_of_interpreters",
    "non_english_speaking_background_flag",
    "custody_served"
]

In [58]:
def exact_match_eval(output_dict, gt_dict, variables_list):
    eval_dict = {}
    for var in variables_list:
        if output_dict[var] == gt_dict[var]:
            eval_dict[var] = 1
        else:
            eval_dict[var] = 0
    return eval_dict

In [60]:
eval_dict_bool = exact_match_eval(output_dict, test1_dict, boolean_variables)
eval_dict_bool

{'mental_disorder_flag': 1,
 'cognitive_disability_flag': 1,
 'preventable_death_flag': 1,
 'use_of_interpreters': 1,
 'non_english_speaking_background_flag': 1,
 'custody_served': 1}

In [62]:
eval_dict_exact_match = exact_match_eval(output_dict, test1_dict, exact_match_variables)
eval_dict_exact_match

{'file_number': 1, 'date_of_findings': 0, 'sex': 1}

In [63]:
output_dict["date_of_findings"]

'21 October 2022'

In [64]:
test1_dict["date_of_findings"]

'21/10/2022'

Important finding for dates: make sure that dates are recorded in a way that is consistent. Date of findings will always be exact. Date of death can vary depending on finding